# Importing Library

In [6]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.layers import Input, Dropout, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomBrightness

# Importing Dataset

In [7]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 123
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

Found 6799 files belonging to 3 classes.
Using 5440 files for training.
Found 6799 files belonging to 3 classes.
Using 1359 files for validation.
Found 2278 files belonging to 3 classes.


# Data Preprocessing

In [8]:
data_augmentation = tf.keras.Sequential([
    RandomRotation(factor=(-0.025, 0.025)),
    RandomFlip(mode="horizontal"),
    RandomBrightness(factor=0.1)
], name="data_augmentation")

# Model Creation

In [9]:
def build_model(input_shape=IMG_SIZE + (3,)):
    base_model = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        input_shape=input_shape,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = Input(shape=input_shape)
    x = data_augmentation(inputs) 
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(3, activation="softmax")(x)

    model = Model(inputs, outputs, name="Human_Emotion_Detection_Model")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=[SparseCategoricalAccuracy(name="accuracy")]
    )
    return model, base_model

model, base_model = build_model()
model.summary()

Model: "Human_Emotion_Detection_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 7, 7, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,335,203 (77.57 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 20,331,360 (77.56 MB)

# Training The Model

In [10]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 41s 84ms/step - accuracy: 0.6364 - loss: 0.8025 - val_accuracy: 0.7174 - val_loss: 0.6572
Epoch 2/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 26s 76ms/step - accuracy: 0.6961 - loss: 0.7004 - val_accuracy: 0.7322 - val_loss: 0.6127
Epoch 3/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 23s 69ms/step - accuracy: 0.7143 - loss: 0.6646 - val_accuracy: 0.7483 - val_loss: 0.5944
Epoch 4/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 26s 76ms/step - accuracy: 0.7276 - loss: 0.6444 - val_accuracy: 0.7557 - val_loss: 0.5737
Epoch 5/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 26s 76ms/step - accuracy: 0.7351 - loss: 0.6335 - val_accuracy: 0.7550 - val_loss: 0.5739


# Fine Tuning

In [11]:
print("Base model has", len(base_model.layers), "layers.")
fine_tune_at = len(base_model.layers) - 50 

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=[SparseCategoricalAccuracy(name="accuracy")]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
    ]
)

Base model has 513 layers.
Epoch 1/10


E0000 00:00:1757245778.923094     714 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/Human_Emotion_Detection_Model_1/efficientnetv2-s_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


340/340 ━━━━━━━━━━━━━━━━━━━━ 52s 111ms/step - accuracy: 0.7072 - loss: 0.6820 - val_accuracy: 0.7653 - val_loss: 0.5768 - learning_rate: 1.0000e-05
Epoch 2/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 30s 89ms/step - accuracy: 0.7412 - loss: 0.6254 - val_accuracy: 0.7734 - val_loss: 0.5409 - learning_rate: 1.0000e-05
Epoch 3/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 32s 94ms/step - accuracy: 0.7500 - loss: 0.5999 - val_accuracy: 0.7815 - val_loss: 0.5154 - learning_rate: 1.0000e-05
Epoch 4/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 29s 86ms/step - accuracy: 0.7625 - loss: 0.5766 - val_accuracy: 0.7873 - val_loss: 0.4971 - learning_rate: 1.0000e-05
Epoch 5/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 30s 90ms/step - accuracy: 0.7627 - loss: 0.5590 - val_accuracy: 0.7925 - val_loss: 0.4856 - learning_rate: 1.0000e-05
Epoch 6/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 32s 93ms/step - accuracy: 0.7767 - loss: 0.5413 - val_accuracy: 0.7999 - val_loss: 0.4695 - learning_rate: 1.0000e-05
Epoch 7/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 31s 92ms/step - accura

# Evaluate

In [ ]:
model.evaluate(test_ds)

143/143 ━━━━━━━━━━━━━━━━━━━━ 7s 50ms/step - accuracy: 0.8134 - loss: 0.4566
Model evaluation on test dataset completed.
